<a href="https://colab.research.google.com/github/vijaygwu/SEAS8525/blob/main/Class_6_TransformerApps_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U ipywidgets nbformat

In [ ]:
text = """Dear Amazon, last week I ordered an Optimus Prime action figure
from your online store in Germany. Unfortunately, when I opened the package,
I discovered to my horror that I had been sent an action figure of Megatron
instead! As a lifelong enemy of the Decepticons, I hope you can understand my
dilemma. To resolve the issue, I demand an exchange of Megatron for the
Optimus Prime figure I ordered. Enclosed are copies of my records concerning
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""


**Text Classification**

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification")

In [ ]:
import pandas as pd

outputs = classifier(text)
pd.DataFrame(outputs)

**Named Entity Recognition**

In [ ]:
ner_tagger = pipeline("ner", aggregation_strategy="simple")
outputs = ner_tagger(text)
pd.DataFrame(outputs)

**Question Answering**

In [ ]:
from PIL import Image, ImageDraw, ImageFont

def text_to_image(text_content, font_size=20, img_width=800, img_height=600):
    """Converts a string of text into a PIL Image."""
    img = Image.new('RGB', (img_width, img_height), color = (255, 255, 255))
    d = ImageDraw.Draw(img)

    # Try to load a common sans-serif font or fallback to default
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", font_size)
    except IOError:
        font = ImageFont.load_default()

    # Wrap text to fit image width
    lines = []
    words = text_content.split(' ')
    current_line = []
    for word in words:
        test_line = ' '.join(current_line + [word])
        bbox = d.textbbox((0,0), test_line, font=font)
        width = bbox[2] - bbox[0]
        if width < img_width - 40: # 40 for padding
            current_line.append(word)
        else:
            lines.append(' '.join(current_line))
            current_line = [word]
    lines.append(' '.join(current_line))

    y_text = 20 # Initial Y position for text
    for line in lines:
        d.text((20, y_text), line, fill=(0, 0, 0), font=font)
        y_text += font_size + 5 # Line spacing

    return img

# Convert the 'text' variable (string) into an image
image_from_text = text_to_image(text)

print("Text successfully converted to image.")
# You can display the image to verify:
# display(image_from_text)

In [ ]:
!apt-get install -y tesseract-ocr
!pip install pytesseract

import pytesseract
# If tesseract is not in your PATH, you might need to specify its path, e.g.:
# pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'

print("Tesseract OCR and pytesseract installed.")

In [ ]:
reader = pipeline("document-question-answering", model="impira/layoutlm-document-qa")
question = "What does the customer want?"
outputs = reader(question=question, image=image_from_text) # Using the generated image
pd.DataFrame([outputs])

In [ ]:
reader = pipeline("document-question-answering", model="impira/layoutlm-document-qa")
question = "What does the customer want?"
outputs = reader(question=question, image=image_from_text) # Now using the generated image
pd.DataFrame([outputs])

**Summarization**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch


model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
summary_ids = model.generate(inputs["input_ids"], max_length=45, min_length=10, num_beams=4)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

**Translation**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-de"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
translated_ids = model.generate(inputs["input_ids"], min_length=100, num_beams=4)
translation = tokenizer.decode(translated_ids[0], skip_special_tokens=True)

print(translation)

**Text Generation**

In [ ]:
generator = pipeline("text-generation")
response = "Dear Bumblebee, I am sorry to hear that your order was mixed up."
prompt = text + "\n\nCustomer service response:\n" + response
outputs = generator(prompt, max_length=200)
print(outputs[0]['generated_text'])